# Desafío Profesional — Data Science
## Caso de Negocio: Cambio Climático
# Etapa 1: Exploración Visual de los Datos (EDA)

---

**Objetivo:** Realizar una exploración visual de los datos utilizando técnicas de análisis exploratorio (EDA) para identificar patrones, tendencias y posibles anomalías.

**Pregunta central del caso:** ¿Está el mundo avanzando lo suficiente en la transición hacia energías renovables para mitigar el cambio climático? ¿Y Argentina en particular?

**Estructura del notebook:**
1. Configuración del entorno y carga de datos
2. Estadísticas descriptivas de cada dataset
3. Visualizaciones exploratorias organizadas por pregunta de investigación
4. Hallazgos y conclusiones

**Datasets utilizados:**

| # | Dataset | Qué contiene | Período |
|---|---------|-------------|----------|
| DS1 | Emisiones CO2 Argentina | Cuánto CO2 emite la red eléctrica argentina por cada MWh generado | 2007-2017 |
| DS2 | Estadísticas energía mundial | Emisiones, electricidad, renovables de ~58 países | 1990-2018 |
| DS3 | PBI per cápita (Banco Mundial) | Cuánto produce económicamente cada país por persona | 1960-2019 |
| DS4 | Población mundial (Banco Mundial) | Cuánta gente vive en cada país | 1960-2019 |
| DS5 | Proyectos renovables Argentina | Cuánta electricidad genera cada central renovable en Argentina | 2011-2019 |

---
## 1. Configuración del entorno

Importamos las librerías de Python necesarias y configuramos el estilo visual de los gráficos.

En términos simples: estamos preparando las "herramientas" que vamos a usar.
- **Pandas** es nuestra herramienta principal para manipular tablas de datos (como un Excel pero mucho más potente).
- **Matplotlib y Seaborn** son las librerías para crear gráficos.
- **NumPy** permite hacer cálculos matemáticos eficientes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración general
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Estilo global de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Paleta de colores personalizada para el proyecto
COLORS = {
    'primary': '#1B4F72',
    'accent': '#2E86C1',
    'success': '#27AE60',
    'warning': '#F39C12',
    'danger': '#E74C3C',
    'gray': '#7F8C8D',
    'argentina': '#75AADB',
}

print('✅ Librerías y configuración cargadas')
print(f'   Pandas: {pd.__version__}')
print(f'   NumPy: {np.__version__}')

### Rutas de archivos

Configuramos dónde están los archivos Excel. Si trabajás en **Google Colab**, ajustá la variable `PATH`.

In [ ]:
# ============================================================
# DETECCIÓN AUTOMÁTICA DE ENTORNO
# ============================================================
# El notebook funciona sin modificación tanto en Google Colab
# como localmente: detecta Colab y, si no, usa la ruta local.
# Si necesitás otra ruta, editá las asignaciones de PATH de abajo.

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    # Montar Google Drive y leer desde la carpeta del proyecto.
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = '/content/drive/MyDrive/desafio_cambio_climatico/datos_originales/'
else:
    # Local: los archivos originales viven en datos/originales/
    # (ver datos/originales/README.md para descargarlos).
    PATH = '../datos/originales/'

FILE_DS1 = PATH + 'Emisio_n_CO2_en_Argentina.xlsx'
FILE_DS2 = PATH + 'Estadísticas_energía_mundial.xlsx'
FILE_DS3 = PATH + 'PBI_per_cápita_-_datos_Banco_Mundial.xls'
FILE_DS4 = PATH + 'Población_mundial_-_dataset_Banco_Mundial.xlsx'
FILE_DS5 = PATH + 'Proyectos_renovables_en_Argentina.xlsx'

print(f'✅ Entorno detectado: {"Google Colab" if EN_COLAB else "Local"}')
print(f'   PATH = {PATH}')

---
## 2. Carga de datos

Cada archivo Excel tiene una estructura de encabezados diferente, lo que requiere un tratamiento específico para cargarlo correctamente. Esto es muy común en proyectos reales de Data Science: los datos rara vez vienen "prolijos".

### Problemas detectados en los archivos originales:

| Dataset | Problema | Cómo lo resolvemos |
|---------|----------|--------------------|
| DS1 | 5 filas de metadata antes de los datos reales | Saltar las primeras 5 filas al leer |
| DS2 | Cada hoja tiene 3 filas de encabezado: título, link, años | Leer sin headers automáticos y extraer los años de la fila 2 |
| DS3/DS4 | 3 filas de metadata del Banco Mundial antes del header real | Saltar 3 filas, tomar la siguiente como header |
| DS5 | Fila 0 completamente vacía, headers reales en fila 1 | Extraer headers de fila 1 manualmente |

Para DS2 y DS3/DS4 creamos **funciones reutilizables** que encapsulan la lógica de parseo, de modo que si en el futuro se agregan más hojas o archivos del Banco Mundial, se puedan cargar con la misma función.

In [ ]:
# ============================================================
# FUNCIONES DE CARGA REUTILIZABLES
# ============================================================

def cargar_hoja_ds2(filepath, sheet_name):
    """
    Carga una hoja del Excel de estadísticas de energía mundial.
    Cada hoja tiene: Fila 0=título, Fila 1='Back to list', Fila 2=años, Fila 3+=datos.
    Retorna un DataFrame limpio y la lista de años disponibles.
    """
    df = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    year_row = df.iloc[2]
    col_names = ['pais_region']
    year_cols = []
    for i in range(1, len(year_row)):
        try:
            year = int(float(year_row.iloc[i]))
            if 1900 < year < 2100:
                col_names.append(year)
                year_cols.append(year)
            else:
                col_names.append(f'extra_{i}')
        except (ValueError, TypeError):
            col_names.append(f'extra_{i}')
    df_data = df.iloc[3:].copy()
    df_data.columns = col_names[:len(df_data.columns)]
    df_data = df_data[[c for c in df_data.columns if not str(c).startswith('extra_')]]
    df_data = df_data.dropna(subset=['pais_region'])
    df_data = df_data[~df_data['pais_region'].str.contains('Source|Copyright|n.a.', na=False)]
    df_data = df_data.reset_index(drop=True)
    for col in year_cols:
        if col in df_data.columns:
            df_data[col] = pd.to_numeric(df_data[col], errors='coerce')
    return df_data, year_cols


def cargar_banco_mundial(filepath):
    """
    Carga un archivo del Banco Mundial (PBI o Población).
    Estructura: 3 filas de metadata, fila 3=headers reales, fila 4+=datos.
    Retorna un DataFrame limpio y la lista de años.
    """
    df = pd.read_excel(filepath, header=None, skiprows=3)
    headers = df.iloc[0].tolist()
    df = df.iloc[1:].copy()
    clean_headers = []
    for h in headers:
        try:
            y = int(float(h))
            clean_headers.append(y if 1900 < y < 2100 else str(h))
        except (ValueError, TypeError):
            clean_headers.append(str(h) if pd.notna(h) else 'drop')
    df.columns = clean_headers
    df = df.drop(columns=['drop'], errors='ignore')
    df = df.dropna(subset=['Country Name']).reset_index(drop=True)
    year_cols = [c for c in df.columns if isinstance(c, int)]
    for y in year_cols:
        df[y] = pd.to_numeric(df[y], errors='coerce')
    return df, year_cols


# Agrupaciones del Banco Mundial que NO son países reales
# (Mundo, BIRF, OCDE, regiones geográficas, categorías de ingreso, etc.)
AGRUPACIONES_BM = {
    'AFE','AFW','ARB','CEB','CSS','EAP','EAR','EAS','ECA','ECS',
    'EMU','EUU','FCS','HIC','HPC','IBD','IBT','IDA','IDB','IDX',
    'INX','LAC','LCN','LDC','LIC','LMC','LMY','LTE','MEA','MIC',
    'MNA','NAC','OED','OSS','PRE','PSS','PST','SAS','SSA','SSF',
    'SST','TEA','TEC','TLA','TMN','TSA','TSS','UMC','WLD'
}

# Agrupaciones en DS2 (regiones/grupos, no países individuales)
AGRUPACIONES_DS2 = ['World', 'OECD', 'G7', 'BRICS', 'Europe', 'European Union',
                    'CIS', 'America', 'North America', 'Latin America', 'Asia',
                    'Pacific', 'Africa', 'Middle-East']

print('✅ Funciones de carga y filtros definidos')

### 2.1 — DS1: Emisiones CO2 en Argentina

Este archivo mide qué tan "sucia" o "limpia" es la electricidad que produce Argentina. Contiene el **factor de emisión de la red eléctrica**, expresado en toneladas de CO2 por cada megawatt-hora generado (tCO2/MWh).

Pensalo así: si el factor es 0.50, significa que por cada MWh de electricidad que produce la red argentina, se lanzan al aire medio tonelada de CO2. Cuanto menor sea este número, más "limpia" es la electricidad.

El archivo tiene dos versiones de este factor:
- **Ex-post:** el valor real medido después de que terminó el año. Es el dato exacto de lo que pasó.
- **Ex-ante:** una proyección calculada como el promedio de los 3 años anteriores. Se usa para planificar proyectos de energía renovable antes de que se construyan (por ejemplo, para estimar cuánto CO2 va a evitar un parque eólico nuevo).

La metodología viene del documento ACM0002 del Mecanismo de Desarrollo Limpio de Naciones Unidas.

In [ ]:
# DS1: Factor de emisión de la red eléctrica argentina
ds1 = pd.read_excel(
    FILE_DS1,
    sheet_name='FACTOR DE EMISION OM SIMPLE',
    skiprows=5, usecols=[1, 2, 3],
    names=['anio', 'factor_expost_tCO2_MWh', 'factor_exante_tCO2_MWh']
)
ds1['anio'] = ds1['anio'].astype(int)
for col in ['factor_expost_tCO2_MWh', 'factor_exante_tCO2_MWh']:
    ds1[col] = pd.to_numeric(ds1[col], errors='coerce')

print('DS1: Emisiones CO2 en Argentina — Factor de emisión')
print('=' * 60)
print(f'Período: {ds1["anio"].min()} - {ds1["anio"].max()} ({len(ds1)} años)')
print(f'Nulos: {ds1.isnull().sum().sum()}')
print(f'\nDatos completos:')
display(ds1)

### 2.2 — DS2: Estadísticas de energía mundial

Este es el dataset más rico del proyecto. Es un Excel con **27 hojas**, cada una con una variable energética diferente medida para ~60 países y regiones entre 1990 y 2018.

De las 27 hojas, seleccionamos las 5 más relevantes para nuestro caso de cambio climático:

| Hoja | Qué mide | Unidad | En términos simples |
|------|----------|--------|---------------------|
| CO2 emissions | Emisiones de CO2 por quema de combustibles | MtCO2 (millones de toneladas) | Cuánto contamina cada país |
| Share of renewables | % de renovables en electricidad | Porcentaje | Qué parte de la electricidad viene de fuentes limpias |
| Electricity production | Producción total de electricidad | TWh (terawatt-hora) | Cuánta electricidad genera cada país |
| Total energy consumption | Consumo total de energía | Mtoe (millones de toneladas equivalentes de petróleo) | Cuánta energía usa cada país en total |
| Share of wind and solar | % de eólica+solar en electricidad | Porcentaje | Cuánta electricidad viene específicamente del viento y el sol |

In [ ]:
# DS2: Estadísticas de energía mundial (5 hojas relevantes)
HOJAS_DS2 = {
    'CO2 emissions from fuel combus': 'co2_emissions_MtCO2',
    'Share of renewables in electri': 'share_renewables_pct',
    'Electricity production':         'electricity_prod_TWh',
    'Total energy consumption':       'energy_consumption_Mtoe',
    'Share of wind and solar in ele': 'share_wind_solar_pct',
}

ds2 = {}
ds2_years = None

print('DS2: Estadísticas de energía mundial')
print('=' * 60)
for hoja, nombre in HOJAS_DS2.items():
    df, years = cargar_hoja_ds2(FILE_DS2, hoja)
    ds2[nombre] = df
    if ds2_years is None: ds2_years = years
    print(f'  ✅ {nombre:<30s} | {df.shape[0]:>3} entidades | {min(years)}-{max(years)}')

print(f'\nPaíses/regiones disponibles ({len(ds2["co2_emissions_MtCO2"])}):')
print(ds2['co2_emissions_MtCO2']['pais_region'].tolist())

### 2.3 — DS3: PBI per cápita (Banco Mundial)

El PBI per cápita mide cuánto produce económicamente un país dividido por su cantidad de habitantes. Se expresa en dólares estadounidenses. Es un indicador del nivel de desarrollo económico: un PBI per cápita alto generalmente implica mayor industria, consumo y — potencialmente — mayor contaminación.

Este dataset nos va a servir para responder la pregunta P2: ¿existe relación entre la riqueza de un país y sus emisiones de CO2?

In [ ]:
# DS3: PBI per cápita
ds3, ds3_years = cargar_banco_mundial(FILE_DS3)

print('DS3: PBI per cápita — Banco Mundial')
print('=' * 60)
print(f'Shape: {ds3.shape} | Período: {min(ds3_years)}-{max(ds3_years)}')
print(f'Indicador: {ds3["Indicator Name"].iloc[0]}')
print(f'\nNulos por década (rango 1990-2018):')
years_analisis = [y for y in ds3_years if 1990 <= y <= 2018]
for d_start in [1990, 2000, 2010]:
    d_end = min(d_start + 9, 2018)
    cols = [y for y in years_analisis if d_start <= y <= d_end]
    nulos = ds3[cols].isnull().sum().sum()
    total = len(ds3) * len(cols)
    print(f'  {d_start}-{d_end}: {nulos:,} nulos de {total:,} ({nulos/total*100:.1f}%)')

print(f'\nArgentina:')
display(ds3[ds3['Country Name']=='Argentina'][['Country Name','Country Code']+[1990,2000,2010,2018]])

### 2.4 — DS4: Población mundial (Banco Mundial)

La población es clave para calcular las emisiones **per cápita** (por persona), que es una medida mucho más justa que las emisiones totales. China emite más CO2 que Dinamarca en total, pero también tiene 240 veces más habitantes. La emisión per cápita pone a todos en igualdad de condiciones.

**Nota importante:** Los archivos del Banco Mundial mezclan 217 países reales con 47 agrupaciones ("Mundo", "BIRF y la AIF", "Ingreso mediano bajo", etc.). Todas usan códigos de 3 letras, así que no se pueden distinguir por formato. Definimos el conjunto `AGRUPACIONES_BM` para filtrarlas correctamente.

In [ ]:
# DS4: Población mundial
ds4, ds4_years = cargar_banco_mundial(FILE_DS4)

# Filtrar: separar países reales de agrupaciones
ds3_paises = ds3[~ds3['Country Code'].isin(AGRUPACIONES_BM)].copy()
ds4_paises = ds4[~ds4['Country Code'].isin(AGRUPACIONES_BM)].copy()

print('DS4: Población mundial — Banco Mundial')
print('=' * 60)
print(f'Total entidades: {len(ds4)}')
print(f'  Países reales: {len(ds4_paises)}')
print(f'  Agrupaciones filtradas: {len(ds4) - len(ds4_paises)}')
print(f'Período: {min(ds4_years)}-{max(ds4_years)}')

print(f'\nTop 10 países más poblados en 2018:')
top_pob = ds4_paises[['Country Name', 2018]].dropna().sort_values(2018, ascending=False).head(10)
top_pob['Población'] = top_pob[2018].apply(lambda x: f'{x/1e6:,.1f}M')
display(top_pob[['Country Name', 'Población']].reset_index(drop=True))

arg_pob = ds4_paises.loc[ds4_paises['Country Name']=='Argentina', 2018].values[0]
print(f'\nArgentina en 2018: {arg_pob/1e6:,.1f} millones de habitantes')

### 2.5 — DS5: Proyectos renovables en Argentina

Este dataset es el más granular: tiene el registro **mensual** de cuánta electricidad genera cada central renovable en Argentina, detallado por máquina, fuente de energía y provincia.

Las fuentes de energía renovable incluidas son: hidroeléctrica (represas pequeñas de hasta 50MW), eólica (molinos de viento), solar (paneles fotovoltaicos), biomasa (residuos orgánicos), biogás y biodiesel.

**⚠️ Atención:** El dataset también contiene registros de "Demanda MEM" (Mercado Eléctrico Mayorista), que es la demanda total de electricidad del país, **no generación renovable**. Hay que separarla del análisis.

In [ ]:
# DS5: Proyectos renovables en Argentina
ds5_raw = pd.read_excel(FILE_DS5, header=None)
ds5 = ds5_raw.iloc[2:].copy()
ds5.columns = ds5_raw.iloc[1].tolist()
ds5 = ds5.rename(columns={
    'AÑO':'anio', 'CENTRAL':'central_cod', 'CENTRAL DESCRIPCIÓN':'central_desc',
    'MAQUINA':'maquina', 'FUENTE DE ENERGÍA':'fuente_energia',
    'REGIÓN':'region', 'PROVINCIA':'provincia', 'MES':'mes',
    'ENERGÍA GENERADA [GWh]':'energia_gwh', 'Nueva Generación':'nueva_generacion'
})
ds5['anio'] = pd.to_numeric(ds5['anio'], errors='coerce').astype('Int64')
ds5['energia_gwh'] = pd.to_numeric(ds5['energia_gwh'], errors='coerce')
ds5['mes'] = pd.to_datetime(ds5['mes'], errors='coerce')
ds5 = ds5.dropna(axis=1, how='all')

# Separar generación renovable de demanda MEM
ds5_renovable = ds5[ds5['fuente_energia'].str.strip() != 'Demanda MEM'].copy()
ds5_demanda = ds5[ds5['fuente_energia'].str.strip() == 'Demanda MEM'].copy()

print('DS5: Proyectos renovables en Argentina')
print('=' * 60)
print(f'Registros totales: {len(ds5):,}')
print(f'  Generación renovable: {len(ds5_renovable):,} registros')
print(f'  Demanda MEM (filtrada): {len(ds5_demanda):,} registros')
print(f'Período: {ds5["anio"].min()} - {ds5["anio"].max()}')
print(f'Centrales: {ds5_renovable["central_cod"].nunique()} | Provincias: {ds5_renovable["provincia"].nunique()}')
print(f'\nFuentes de energía renovable:')
print(ds5_renovable['fuente_energia'].dropna().unique().tolist())
print(f'\nGeneración por fuente (GWh total):')
display(ds5_renovable.groupby('fuente_energia')['energia_gwh'].sum().sort_values(ascending=False).round(1))

### Checkpoint de carga

Verificamos que todos los datasets están cargados correctamente antes de avanzar con las estadísticas y visualizaciones.

In [ ]:
print('=' * 70)
print('RESUMEN DE CARGA — TODOS LOS DATASETS')
print('=' * 70)
print(f'  DS1 (CO2 Argentina):     {ds1.shape[0]:>6} filas   | {ds1["anio"].min()}-{ds1["anio"].max()}  | Nulos: {ds1.isnull().sum().sum()}')
print(f'  DS2 (Energía mundial):   {len(ds2):>6} hojas  | {min(ds2_years)}-{max(ds2_years)} | {ds2["co2_emissions_MtCO2"].shape[0]} entidades')
print(f'  DS3 (PBI per cápita):    {ds3_paises.shape[0]:>6} países | {min(ds3_years)}-{max(ds3_years)} | (filtrados de {ds3.shape[0]} entidades)')
print(f'  DS4 (Población):         {ds4_paises.shape[0]:>6} países | {min(ds4_years)}-{max(ds4_years)} | (filtrados de {ds4.shape[0]} entidades)')
print(f'  DS5 (Renovables Arg):    {ds5_renovable.shape[0]:>6} filas   | {ds5["anio"].min()}-{ds5["anio"].max()}  | (sin Demanda MEM)')
print(f'\n✅ Todos los datasets cargados correctamente')
print(f'\n📌 Rango temporal común para análisis global: 1990-2018')
print(f'📌 Rango temporal Argentina específico: 2011-2017 (solapamiento DS1 + DS5)')

---
## 3. Estadísticas descriptivas

Antes de graficar, necesitamos entender los números: rangos, promedios, valores extremos, datos faltantes. Esto nos ayuda a detectar problemas en los datos y a formarnos una primera idea de las respuestas a nuestras preguntas.

In [ ]:
# DS1: Estadísticas del factor de emisión
print('DS1: Factor de emisión — Estadísticas')
print('-' * 60)
display(ds1.describe())
print(f'\nTendencia: el factor ex-post pasó de {ds1["factor_expost_tCO2_MWh"].iloc[0]:.4f} ({ds1["anio"].iloc[0]}) a {ds1["factor_expost_tCO2_MWh"].iloc[-1]:.4f} ({ds1["anio"].iloc[-1]})')
print(f'Máximo: {ds1["factor_expost_tCO2_MWh"].max():.4f} en {ds1.loc[ds1["factor_expost_tCO2_MWh"].idxmax(), "anio"]}')
print(f'→ La red eléctrica argentina se fue "limpiando" a partir de 2012')

In [ ]:
# DS2: Top 10 emisores de CO2 en 2018
co2 = ds2['co2_emissions_MtCO2']
co2_paises = co2[~co2['pais_region'].isin(AGRUPACIONES_DS2)]

print('DS2: Top 10 emisores de CO2 en 2018 (MtCO2)')
print('-' * 60)
top10 = co2_paises[['pais_region', 2018]].dropna().sort_values(2018, ascending=False).head(10)
display(top10.reset_index(drop=True))

# Estadísticas por hoja
print(f'\nResumen por variable (2018):')
for nombre, df in ds2.items():
    if 2018 in df.columns:
        vals = df[2018].dropna()
        print(f'  {nombre}: media={vals.mean():.1f}, mediana={vals.median():.1f}, min={vals.min():.1f}, max={vals.max():.1f}')

In [ ]:
# DS3/DS4: Argentina en contexto
arg_pbi = ds3_paises.loc[ds3_paises['Country Name']=='Argentina', 2018].values[0]
percentil_pbi = (ds3_paises[2018].dropna() < arg_pbi).mean() * 100

print('DS3/DS4: Argentina en contexto mundial (2018)')
print('-' * 60)
print(f'PBI per cápita: US$ {arg_pbi:,.0f} (percentil {percentil_pbi:.0f} entre {len(ds3_paises)} países)')
print(f'Población: {arg_pob/1e6:,.1f} millones')
print(f'\n→ Argentina está en el tercio superior de ingresos a nivel mundial')

---
## 4. Visualizaciones exploratorias

Las visualizaciones están organizadas siguiendo las **4 preguntas guía** del caso de negocio:

- **P1:** ¿Cómo ha evolucionado la emisión de CO2 y qué patrones pueden predecirse?
- **P2:** ¿Cuál es la relación entre emisiones, PBI y población?
- **P3:** ¿Qué patrones emergen de las energías renovables entre países?
- **P4:** ¿Cómo se compara Argentina con el resto del mundo?

### 4.1 — Evolución global de emisiones de CO2 (P1)

Empezamos con la pregunta más básica: ¿el mundo está emitiendo más o menos CO2 con el paso del tiempo? Y si está emitiendo más, ¿quién es responsable?

In [ ]:
co2 = ds2['co2_emissions_MtCO2']
regiones = ['World', 'OECD', 'BRICS', 'Europe', 'Latin America']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo: Evolución por región
ax = axes[0]
for region in regiones:
    row = co2[co2['pais_region'] == region]
    if not row.empty:
        valores = row[ds2_years].values.flatten()
        lw = 2.5 if region == 'World' else 1.5
        ax.plot(ds2_years, valores / 1000, label=region, linewidth=lw)
ax.set_title('Emisiones de CO2 por región (1990-2018)')
ax.set_xlabel('Año')
ax.set_ylabel('GtCO2 (gigatoneladas)')
ax.legend(loc='upper left')
ax.set_xlim(1990, 2018)

# Panel derecho: Top 10 países emisores en 2018
ax = axes[1]
top10_co2 = co2_paises[['pais_region', 2018]].dropna().sort_values(2018, ascending=True).tail(10)
colors = [COLORS['argentina'] if p == 'Argentina' else COLORS['primary'] for p in top10_co2['pais_region']]
ax.barh(top10_co2['pais_region'], top10_co2[2018], color=colors)
ax.set_title('Top 10 países emisores de CO2 (2018)')
ax.set_xlabel('MtCO2')
for i, (_, row) in enumerate(top10_co2.iterrows()):
    ax.text(row[2018] + 50, i, f'{row[2018]:,.0f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

w90 = co2[co2['pais_region']=='World'][1990].values[0]
w18 = co2[co2['pais_region']=='World'][2018].values[0]
print(f'📊 Las emisiones mundiales crecieron {((w18/w90)-1)*100:.1f}% entre 1990 y 2018')
print(f'   De {w90/1000:.1f} GtCO2 a {w18/1000:.1f} GtCO2')

### 4.2 — Emisiones de CO2 per cápita (P1 + P2)

Las emisiones totales no cuentan toda la historia. Un país gigante como China emite mucho en total simplemente porque tiene 1,400 millones de habitantes. La medida más justa es **cuánto emite cada persona**: las emisiones per cápita.

Para calcularlas, tomamos las emisiones totales de cada país (DS2) y las dividimos por su población (DS4). Esto requiere cruzar dos datasets distintos, un anticipo del trabajo de integración de la Etapa 2.

In [ ]:
# Mapeo de nombres DS2 → DS4 para países que difieren
MAPEO_NOMBRES = {
    'United States': 'Estados Unidos', 'United Kingdom': 'Reino Unido',
    'South Korea': 'Corea, República de', 'Russia': 'Federación de Rusia',
    'Iran': 'Irán, República Islámica del', 'Venezuela': 'Venezuela, RB',
    'Egypt': 'Egipto, República Árabe de', 'Czech Republic': 'Chequia',
    'Turkey': 'Türkiye',
}

co2_solo = co2[~co2['pais_region'].isin(AGRUPACIONES_DS2)].copy()
percapita_data = []
for _, row in co2_solo.iterrows():
    pais_ds2 = row['pais_region']
    pais_ds4 = MAPEO_NOMBRES.get(pais_ds2, pais_ds2)
    pob_row = ds4_paises[ds4_paises['Country Name'] == pais_ds4]
    if not pob_row.empty and pd.notna(row[2018]) and pd.notna(pob_row[2018].values[0]):
        pob = pob_row[2018].values[0]
        if pob > 0:
            percapita_data.append({'pais': pais_ds2, 'co2_total_Mt': row[2018],
                                   'poblacion': pob, 'co2_percapita_t': (row[2018]*1e6)/pob})

df_percapita = pd.DataFrame(percapita_data).sort_values('co2_percapita_t', ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))
colors = [COLORS['argentina'] if p == 'Argentina' else COLORS['accent'] for p in df_percapita['pais']]
ax.barh(df_percapita['pais'], df_percapita['co2_percapita_t'], color=colors)
ax.set_title('Emisiones de CO2 per cápita por país (2018)')
ax.set_xlabel('tCO2 por persona')
med = df_percapita['co2_percapita_t'].median()
ax.axvline(x=med, color=COLORS['danger'], linestyle='--', alpha=0.7, label=f'Mediana: {med:.1f} t')
ax.legend()
plt.tight_layout()
plt.show()

arg_pc = df_percapita[df_percapita['pais']=='Argentina']['co2_percapita_t'].values
if len(arg_pc) > 0:
    print(f'📊 Argentina emite {arg_pc[0]:.1f} tCO2 per cápita — cerca de la mediana mundial')

### 4.3 — Heatmap de correlaciones (P2)

Un **heatmap de correlaciones** muestra qué tan relacionadas están las variables entre sí. Los valores van de -1 (relación inversa perfecta) a +1 (relación directa perfecta). Un valor cercano a 0 indica que no hay relación lineal.

Esto nos anticipa qué variables serán útiles como predictores en los modelos de regresión de la Etapa 3. Si dos variables están muy correlacionadas entre sí (por ejemplo, población y CO2 total), puede haber **multicolinealidad**, que complica los modelos.

In [ ]:
renew = ds2['share_renewables_pct']
elec = ds2['electricity_prod_TWh']
energy = ds2['energy_consumption_Mtoe']

corr_data = []
for _, row in co2_solo.iterrows():
    pais = row['pais_region']
    pais_bm = MAPEO_NOMBRES.get(pais, pais)
    pob_r = ds4_paises[ds4_paises['Country Name']==pais_bm]
    pbi_r = ds3_paises[ds3_paises['Country Name']==pais_bm]
    ren_r = renew[renew['pais_region']==pais]
    elec_r = elec[elec['pais_region']==pais]
    energy_r = energy[energy['pais_region']==pais]
    
    if not pob_r.empty and not pbi_r.empty:
        pob_val = pob_r[2018].values[0]
        co2_pc = (row[2018]*1e6/pob_val) if pob_val and pob_val > 0 else np.nan
        corr_data.append({
            'pais': pais,
            'CO2 (MtCO2)': row[2018],
            'CO2 per cápita (t)': co2_pc,
            'PBI per cápita (US$)': pbi_r[2018].values[0],
            'Población (M)': pob_val/1e6 if pob_val else np.nan,
            '% Renovables': ren_r[2018].values[0] if not ren_r.empty else np.nan,
            'Electricidad (TWh)': elec_r[2018].values[0] if not elec_r.empty else np.nan,
            'Energía (Mtoe)': energy_r[2018].values[0] if not energy_r.empty else np.nan,
        })

df_corr = pd.DataFrame(corr_data)
numeric_cols = [c for c in df_corr.columns if c != 'pais']

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df_corr[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Matriz de correlación — Variables clave (2018)')
plt.tight_layout()
plt.show()

print('📊 Correlaciones más fuertes con CO2 total:')
for var, val in corr_matrix['CO2 (MtCO2)'].drop('CO2 (MtCO2)').sort_values(key=abs, ascending=False).items():
    print(f'   {var}: {val:.3f}')


### 4.4 — CO2 per cápita vs PBI per cápita (P2)

Este gráfico es uno de los más reveladores del proyecto. Cada burbuja es un país. La posición horizontal muestra su riqueza (PBI per cápita) y la vertical su contaminación por persona (CO2 per cápita). El **tamaño** de la burbuja refleja la población, y el **color** indica qué porcentaje de su electricidad viene de renovables (verde = más renovables, rojo = menos).

La pregunta que responde: ¿es inevitable que un país rico contamine mucho? ¿O se puede ser rico y limpio?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
df_plot = df_corr.dropna(subset=['CO2 per cápita (t)', 'PBI per cápita (US$)', 'Población (M)'])
sizes = df_plot['Población (M)'] / df_plot['Población (M)'].max() * 500 + 20
scatter = ax.scatter(df_plot['PBI per cápita (US$)'], df_plot['CO2 per cápita (t)'],
                     s=sizes, c=df_plot['% Renovables'], cmap='RdYlGn',
                     alpha=0.7, edgecolors='gray', linewidth=0.5)
cbar = plt.colorbar(scatter, ax=ax, shrink=0.8)
cbar.set_label('% Renovables en electricidad')

for _, row in df_plot.iterrows():
    if row['pais'] in ['Argentina','China','United States','India','Brazil','Germany','Saudi Arabia','Japan','Australia']:
        ax.annotate(row['pais'], (row['PBI per cápita (US$)'], row['CO2 per cápita (t)']),
                    fontsize=9, ha='left', va='bottom', xytext=(5,5), textcoords='offset points')

ax.set_title('CO2 per cápita vs PBI per cápita (2018)\nTamaño = población, Color = % renovables')
ax.set_xlabel('PBI per cápita (US$)')
ax.set_ylabel('CO2 per cápita (tCO2/persona)')
ax.set_xlim(left=0); ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

r = df_plot[['CO2 per cápita (t)', 'PBI per cápita (US$)']].corr().iloc[0,1]
print(f'📊 Correlación CO2 per cápita vs PBI per cápita: {r:.3f}')
print(f'→ Hay relación positiva, pero con excepciones notables (países verdes arriba a la derecha = ricos pero limpios)')

### 4.5 — Energías renovables: evolución y ranking de países (P3)

¿Está el mundo migrando hacia energías limpias? ¿Quiénes lideran y quiénes se quedan atrás?

El concepto de "renovable" incluye hidroeléctrica, eólica, solar, biomasa, geotérmica y mareomotriz. Es importante notar que muchos países latinoamericanos tienen un alto porcentaje de renovables gracias a sus represas hidroeléctricas, que llevan décadas funcionando. El desafío moderno es el crecimiento de eólica y solar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for region in ['World', 'Europe', 'OECD', 'BRICS', 'Latin America']:
    row = renew[renew['pais_region']==region]
    if not row.empty:
        ax.plot(ds2_years, row[ds2_years].values.flatten(),
                label=region, linewidth=2.5 if region=='World' else 1.5)
ax.set_title('% Renovables en electricidad por región (1990-2018)')
ax.set_xlabel('Año'); ax.set_ylabel('% Renovables'); ax.legend(); ax.set_xlim(1990, 2018)

ax = axes[1]
renew_p = renew[~renew['pais_region'].isin(AGRUPACIONES_DS2)]
r2018 = renew_p[['pais_region', 2018]].dropna().sort_values(2018, ascending=True)
bottom5 = r2018.head(5); top5 = r2018.tail(5)
arg_r = r2018[r2018['pais_region']=='Argentina']
to_plot = pd.concat([bottom5, arg_r, top5]).drop_duplicates().sort_values(2018)
colors = [COLORS['argentina'] if p=='Argentina' else
          (COLORS['success'] if v>50 else COLORS['danger'] if v<15 else COLORS['warning'])
          for p,v in zip(to_plot['pais_region'], to_plot[2018])]
ax.barh(to_plot['pais_region'], to_plot[2018], color=colors)
ax.set_title('% Renovables — Selección de países (2018)')
ax.set_xlabel('% Renovables')
for i, (_,row) in enumerate(to_plot.iterrows()):
    ax.text(row[2018]+0.5, i, f'{row[2018]:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 4.6 — El boom de la eólica y la solar (P3)

Si separamos la eólica y la solar del resto de las renovables (que incluyen la hidroeléctrica, que ya era importante en los 90), se revela una historia diferente: un **crecimiento exponencial** que arrancó alrededor de 2010 y se aceleró en la última década.

In [ ]:
ws = ds2['share_wind_solar_pct']
fig, ax = plt.subplots(figsize=(14, 6))
for region in ['World', 'Europe', 'OECD', 'BRICS', 'Latin America']:
    row = ws[ws['pais_region']==region]
    if not row.empty:
        ax.plot(ds2_years, row[ds2_years].values.flatten(),
                label=region, linewidth=2.5 if region=='World' else 1.5, marker='o', markersize=3)
ax.set_title('% Eólica + Solar en electricidad (1990-2018)')
ax.set_xlabel('Año'); ax.set_ylabel('% Eólica + Solar'); ax.legend(); ax.set_xlim(1990, 2018)
plt.tight_layout()
plt.show()

ws10 = ws[ws['pais_region']=='World'][2010].values[0]
ws18 = ws[ws['pais_region']=='World'][2018].values[0]
print(f'📊 Eólica+Solar mundial: {ws10:.1f}% (2010) → {ws18:.1f}% (2018)')
print(f'   Europa lidera con {ws[ws["pais_region"]=="Europe"][2018].values[0]:.1f}% en 2018')

### 4.7 — Foco Argentina: factor de emisión y generación renovable (P4)

Estos dos gráficos cuentan una historia conectada.

**El gráfico de la izquierda** muestra qué tan "sucia" es la electricidad argentina. Cada punto es un año. Cuanto más alto, más CO2 se emite por cada MWh generado. Se ve que entre 2007 y 2012 la red se ensució (más centrales térmicas), pero a partir de 2012 empezó a limpiarse.

**El gráfico de la derecha** explica por qué: muestra cuánta electricidad renovable se genera por fuente. Hasta 2016 casi todo venía de represas pequeñas (barras celestes). Pero en 2017-2019 aparecen con fuerza la eólica (verde) y la solar (amarilla). Esa incorporación de renovables coincide con la baja del factor de emisión.

La conexión: a medida que Argentina suma más generación renovable, cada MWh de su red eléctrica contamina menos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Factor de emisión
ax = axes[0]
ax.plot(ds1['anio'], ds1['factor_expost_tCO2_MWh'], 'o-',
        color=COLORS['danger'], label='Ex-post (real)', linewidth=2, markersize=6)
ax.plot(ds1['anio'], ds1['factor_exante_tCO2_MWh'], 's--',
        color=COLORS['warning'], label='Ex-ante (proyección)', linewidth=2, markersize=6)
ax.fill_between(ds1['anio'], ds1['factor_expost_tCO2_MWh'], alpha=0.1, color=COLORS['danger'])
ax.set_title('Argentina — Factor de emisión de la red eléctrica')
ax.set_xlabel('Año'); ax.set_ylabel('tCO2/MWh'); ax.legend(); ax.set_ylim(0.45, 0.60)

# Generación renovable por fuente
ax = axes[1]
gen_anual = ds5_renovable.groupby(['anio','fuente_energia'])['energia_gwh'].sum().reset_index()
gen_pivot = gen_anual.pivot_table(index='anio', columns='fuente_energia', values='energia_gwh', aggfunc='sum').fillna(0)
fuente_colors = {'HIDRO <=50MW':'#3498DB', 'EOLICO':'#2ECC71', 'BIOMASA':'#8E44AD',
                 'SOLAR':'#F1C40F', 'BIOGAS':'#E67E22', 'BIODIESEL':'#E74C3C'}
gen_pivot.plot(kind='bar', stacked=True, ax=ax,
               color=[fuente_colors.get(c, '#95A5A6') for c in gen_pivot.columns], width=0.8)
ax.set_title('Argentina — Generación renovable por fuente (GWh)')
ax.set_xlabel('Año'); ax.set_ylabel('GWh')
ax.legend(title='Fuente', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

t18 = gen_pivot.loc[2018].sum() if 2018 in gen_pivot.index else 0
t11 = gen_pivot.loc[2011].sum() if 2011 in gen_pivot.index else 0
if t11 > 0:
    print(f'📊 Generación renovable Argentina: {t11:.0f} GWh (2011) → {t18:.0f} GWh (2018) — Crecimiento: {((t18/t11)-1)*100:.1f}%')

### 4.8 — Argentina vs Latinoamérica (P4)

¿Cómo se posiciona Argentina respecto a sus vecinos? Comparamos la evolución de emisiones y renovables con Brasil, Chile, Colombia y México.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
paises_ref = ['Argentina', 'Brazil', 'Chile', 'Colombia', 'Mexico']

for pais in paises_ref:
    row = co2[co2['pais_region']==pais]
    if not row.empty:
        lw = 2.5 if pais=='Argentina' else 1.2
        st = '-' if pais=='Argentina' else '--'
        axes[0].plot(ds2_years, row[ds2_years].values.flatten(), st, label=pais, linewidth=lw)
    row2 = renew[renew['pais_region']==pais]
    if not row2.empty:
        axes[1].plot(ds2_years, row2[ds2_years].values.flatten(), st, label=pais, linewidth=lw)

axes[0].set_title('Emisiones CO2 — Argentina vs Latinoamérica')
axes[0].set_xlabel('Año'); axes[0].set_ylabel('MtCO2'); axes[0].legend(); axes[0].set_xlim(1990, 2018)
axes[1].set_title('% Renovables — Argentina vs Latinoamérica')
axes[1].set_xlabel('Año'); axes[1].set_ylabel('% Renovables'); axes[1].legend(); axes[1].set_xlim(1990, 2018)

plt.tight_layout()
plt.show()

### 4.9 — Demanda MEM: ¿cuánto de lo que consume Argentina viene de renovables? (P4)

Hasta ahora separamos los datos de "Demanda MEM" del análisis renovable porque no es generación. Pero esos datos son muy valiosos: representan **cuánta electricidad consumió todo el país** mes a mes.

Al cruzar la demanda total con la generación renovable, podemos responder una pregunta clave: **¿qué porcentaje de la electricidad que consume Argentina proviene de fuentes renovables?** Esto es la **penetración real** de las renovables en la matriz eléctrica.

**¿Qué es el MEM?** El Mercado Eléctrico Mayorista es donde los grandes generadores (centrales térmicas, hidroeléctricas, parques eólicos y solares) venden la electricidad que producen. Las distribuidoras la compran ahí y la llevan a hogares, comercios e industrias. La "Demanda MEM" es entonces el total de electricidad que el país necesitó en un período dado.

In [ ]:
# Preparación: Demanda MEM vs Generación Renovable
ds5_mem = ds5[ds5['fuente_energia'].str.strip() == 'Demanda MEM'].copy()

mem_anual = ds5_mem.groupby('anio')['energia_gwh'].sum().reset_index()
mem_anual.columns = ['anio', 'demanda_gwh']
renov_anual = ds5_renovable.groupby('anio')['energia_gwh'].sum().reset_index()
renov_anual.columns = ['anio', 'renovable_gwh']

ratio_anual = mem_anual.merge(renov_anual, on='anio')
ratio_anual['ratio_pct'] = (ratio_anual['renovable_gwh'] / ratio_anual['demanda_gwh']) * 100
ratio_anual['no_renovable_gwh'] = ratio_anual['demanda_gwh'] - ratio_anual['renovable_gwh']

# Datos mensuales 2018
ds5_mem_2018 = ds5_mem[ds5_mem['anio']==2018].copy()
ds5_mem_2018['mes_num'] = ds5_mem_2018['mes'].dt.month
ds5_ren_2018 = ds5_renovable[ds5_renovable['anio']==2018].copy()
ds5_ren_2018['mes_num'] = ds5_ren_2018['mes'].dt.month
mem_mensual = ds5_mem_2018.groupby('mes_num')['energia_gwh'].sum()
ren_mensual = ds5_ren_2018.groupby('mes_num')['energia_gwh'].sum()

print('Penetración renovable en la demanda eléctrica argentina:')
print('=' * 70)
for _, row in ratio_anual.iterrows():
    barra = '█' * int(row['ratio_pct'] * 4)
    print(f'  {int(row["anio"])}: {barra} {row["ratio_pct"]:.2f}%  ({row["renovable_gwh"]:,.0f} de {row["demanda_gwh"]:,.0f} GWh)')


#### Evolución anual del ratio renovable/demanda

Este gráfico muestra la **penetración real** de las renovables: qué porcentaje de toda la electricidad que consumió Argentina fue generada por fuentes renovables.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ratio_plot = ratio_anual[ratio_anual['anio'] <= 2018]
ax.plot(ratio_plot['anio'], ratio_plot['ratio_pct'], 'o-',
        color=COLORS['success'], linewidth=2.5, markersize=8)
ax.fill_between(ratio_plot['anio'], ratio_plot['ratio_pct'], alpha=0.15, color=COLORS['success'])
for _, row in ratio_plot.iterrows():
    ax.annotate(f'{row["ratio_pct"]:.2f}%', (row['anio'], row['ratio_pct']),
                textcoords='offset points', xytext=(0, 12), ha='center', fontsize=10)
ax.set_title('Penetración renovable en la demanda eléctrica argentina')
ax.set_xlabel('Año'); ax.set_ylabel('% de la demanda cubierto por renovables')
ax.set_ylim(0, max(ratio_plot['ratio_pct']) * 1.4)

ax = axes[1]
ax.bar(ratio_plot['anio'], ratio_plot['renovable_gwh'], 0.6,
       label='Renovable', color=COLORS['success'], alpha=0.85)
ax.bar(ratio_plot['anio'], ratio_plot['no_renovable_gwh'], 0.6,
       bottom=ratio_plot['renovable_gwh'],
       label='No renovable (térmico, hidro grande, nuclear)', color=COLORS['gray'], alpha=0.5)
ax.set_title('Composición de la demanda eléctrica argentina (GWh)')
ax.set_xlabel('Año'); ax.set_ylabel('GWh'); ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

r11 = ratio_plot[ratio_plot['anio']==2011]['ratio_pct'].values[0]
r18 = ratio_plot[ratio_plot['anio']==2018]['ratio_pct'].values[0]
print(f'📊 La penetración renovable se duplicó: {r11:.2f}% (2011) → {r18:.2f}% (2018)')
print(f'   Pero aún representa una fracción pequeña de la demanda total')


#### Demanda vs generación renovable — patrón mensual (2018)

Este gráfico superpone dos cosas para 2018: la **demanda total** (barras grises) con su patrón estacional (picos en invierno y verano), y la **generación renovable** (línea verde) que fue creciendo mes a mes a medida que se conectaban nuevos parques. Lo interesante es que la renovable crece incluso cuando la demanda baja, porque el crecimiento no depende del consumo sino de la **incorporación de nueva capacidad instalada**.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))
meses_labels = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

ax1.bar(range(1,13), mem_mensual.values, color=COLORS['gray'], alpha=0.4,
        label='Demanda total MEM', width=0.6)
ax1.set_xlabel('Mes (2018)'); ax1.set_ylabel('Demanda total (GWh)', color=COLORS['gray'])
ax1.set_xticks(range(1,13)); ax1.set_xticklabels(meses_labels)

ax2 = ax1.twinx()
ax2.plot(range(1,13), ren_mensual.values, 'o-',
         color=COLORS['success'], linewidth=2.5, markersize=8, label='Generación renovable')
ax2.set_ylabel('Generación renovable (GWh)', color=COLORS['success'])

for m in range(1,13):
    r = ren_mensual.get(m,0)/mem_mensual.get(m,1)*100
    ax2.annotate(f'{r:.1f}%', (m, ren_mensual.get(m,0)),
                 textcoords='offset points', xytext=(0,12), ha='center',
                 fontsize=9, color=COLORS['success'], fontweight='bold')

ax1.set_title('Demanda total vs Generación renovable — Mensual 2018\n'
              'Los porcentajes indican la participación renovable en cada mes')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.show()

print(f'📊 La generación renovable se duplicó dentro del año 2018:')
print(f'   Enero: {ren_mensual.get(1,0):.0f} GWh (1.95%) → Diciembre: {ren_mensual.get(12,0):.0f} GWh (4.52%)')


#### La triple relación: ratio renovable, factor de emisión y demanda

Este es el gráfico que cierra la historia argentina. Superpone tres variables en el período 2011-2017 (donde se solapan DS1 y DS5):
- **Factor de emisión** (línea roja): cuánto CO2 emite cada MWh de la red.
- **Participación renovable** (línea verde): qué porcentaje viene de renovables.
- **Demanda total** (barras grises de fondo): el consumo del país.

Si la hipótesis es correcta — las renovables ayudan a mitigar el cambio climático — deberíamos ver que cuando sube la verde, baja la roja.

In [ ]:
triple = ratio_anual.merge(ds1[['anio','factor_expost_tCO2_MWh']], on='anio', how='inner')
triple = triple[triple['anio'] <= 2018]

fig, ax1 = plt.subplots(figsize=(14, 7))

ax1.bar(triple['anio'], triple['demanda_gwh'], color=COLORS['gray'], alpha=0.15, width=0.5, zorder=1)
ax1.set_ylabel('Demanda total (GWh)', color=COLORS['gray'])
ax1.set_ylim(0, 200000)

ax_fe = ax1.twinx()
line1 = ax_fe.plot(triple['anio'], triple['factor_expost_tCO2_MWh'], 'o-',
                   color=COLORS['danger'], linewidth=2.5, markersize=8, label='Factor emisión (tCO2/MWh)', zorder=3)
ax_fe.set_ylabel('Factor de emisión (tCO2/MWh)', color=COLORS['danger'])
ax_fe.set_ylim(0.45, 0.60)

ax_r = ax1.twinx()
ax_r.spines['right'].set_position(('outward', 60))
line2 = ax_r.plot(triple['anio'], triple['ratio_pct'], 's-',
                  color=COLORS['success'], linewidth=2.5, markersize=8, label='% Renovable en demanda', zorder=3)
ax_r.set_ylabel('% Renovable en demanda', color=COLORS['success'])
ax_r.set_ylim(0, 4)

for _, row in triple.iterrows():
    ax_fe.annotate(f'{row["factor_expost_tCO2_MWh"]:.3f}', (row['anio'], row['factor_expost_tCO2_MWh']),
                   textcoords='offset points', xytext=(-10,12), fontsize=9, color=COLORS['danger'])
    ax_r.annotate(f'{row["ratio_pct"]:.2f}%', (row['anio'], row['ratio_pct']),
                  textcoords='offset points', xytext=(10,-15), fontsize=9, color=COLORS['success'])

ax1.set_title('Argentina — Triple relación: Factor de emisión, Participación renovable y Demanda\n'
              'Cuando sube la participación renovable (verde), baja el factor de emisión (rojo)', fontsize=13)
ax1.set_xlabel('Año')
lines = line1 + line2
ax1.legend(lines, [l.get_label() for l in lines], loc='upper center', ncol=2, fontsize=11)
plt.tight_layout()
plt.show()

corr_fe_renov = triple[['factor_expost_tCO2_MWh','ratio_pct']].corr().iloc[0,1]
print(f'📊 Correlación factor de emisión vs % renovable: {corr_fe_renov:.3f}')
if corr_fe_renov < -0.3:
    print(f'   → Relación inversa confirmada: más renovables = menos CO2 por MWh')
else:
    print(f'   → La relación no es tan clara (el factor también depende del mix térmico)')


---
## 5. Hallazgos y conclusiones de la Etapa 1

### Sobre las emisiones de CO2 (P1):
- Las emisiones mundiales crecieron significativamente entre 1990 y 2018, impulsadas por los BRICS.
- Europa y la OCDE estabilizaron o redujeron sus emisiones.
- Los mayores emisores per cápita son los países petroleros del Golfo y Australia.

### Sobre la relación CO2 — PBI — Población (P2):
- Correlación positiva entre PBI per cápita y CO2 per cápita, con excepciones importantes.
- Los países con alto % renovable contaminan menos por persona para un mismo nivel de riqueza.

### Sobre energías renovables (P3):
- El % global de renovables creció, liderado por Europa.
- Eólica y solar muestran crecimiento exponencial desde 2010.
- Latinoamérica tiene alta participación gracias a represas históricas, pero eólica/solar crecen más lento.

### Sobre Argentina (P4):
- El factor de emisión bajó de 0.585 (2012) a 0.480 (2017) — la red se está limpiando.
- La generación renovable creció 140% entre 2011 y 2018.
- La **penetración renovable en la demanda total (MEM)** pasó de 1.21% (2011) a 2.54% (2018). Aún pequeña, pero con crecimiento acelerado.
- Dentro de 2018, la participación mensual se duplicó de enero (1.95%) a diciembre (4.52%), reflejando la conexión progresiva de parques del programa RenovAr.
- La **triple relación** confirma que a medida que crece la proporción de renovables, baja la intensidad de CO2 de la red eléctrica.

### Problemas detectados para la Etapa 2:
- Banco Mundial mezcla países con agrupaciones (resuelto con `AGRUPACIONES_BM`).
- DS5: "Demanda MEM" fue aprovechada para el análisis de penetración renovable.
- Nombres de países difieren entre datasets (resuelto parcialmente con `MAPEO_NOMBRES`).
- 2019 en DS5 tiene datos incompletos.
- Falta integrar formalmente todos los datasets en un dataset maestro.

### Próximo paso:
**Etapa 2 — Limpieza y transformación:** integrar los 5 datasets en un dataset maestro global y uno específico de Argentina.